# Entraînement du Modèle Prédictif - Nutri-Predict

Ce notebook présente la phase d'entraînement du modèle de Machine Learning pour la prédiction de la variation pondérale (`weight_delta_grams`) en fonction du métabolisme de base, de l'apport calorique du biscuit et de la portion consommée.

## 1. Importation des librairies

Nous commençons par importer les librairies nécessaires : DuckDB pour la lecture rapide, Pandas pour la manipulation de données, Scikit-Learn pour la modélisation et l'évaluation, et Joblib pour la sérialisation du modèle.

In [2]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

## 2. Chargement du dataset d'entraînement

Les données générées lors de l'étape de Feature Engineering sont chargées à partir du fichier Parquet `ml_training_dataset.parquet` en utilisant DuckDB.

In [3]:
df_ml = duckdb.query("SELECT * FROM '../data/processed/ml_training_dataset.parquet'").to_df()
df_ml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_bmr               5000 non-null   float64
 1   biscuit_calories_100g  5000 non-null   float64
 2   portion_g              5000 non-null   float64
 3   calories_consommees    5000 non-null   float64
 4   balance_calorique      5000 non-null   float64
 5   weight_delta_grams     5000 non-null   float64
dtypes: float64(6)
memory usage: 234.5 KB


## 3. Séparation des Features et de la Target (Split)

Nous définissons les variables explicatives `X` (le BMR de l'utilisateur, les calories aux 100g du biscuit et la portion consommée) et la variable cible `y` (la variation de poids en grammes). Nous effectuons ensuite une séparation en jeux d'entraînement et de test ($80/20$) avec un état aléatoire fixé pour assurer la reproductibilité.

In [4]:
X = df_ml[['user_bmr', 'biscuit_calories_100g', 'portion_g']]
y = df_ml['weight_delta_grams']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Taille du jeu d'entraînement : {X_train.shape[0]} échantillons")
print(f"Taille du jeu de test : {X_test.shape[0]} échantillons")

Taille du jeu d'entraînement : 4000 échantillons
Taille du jeu de test : 1000 échantillons


## 4. Entraînement du modèle RandomForestRegressor

Nous utilisons un algorithme d'ensemble non linéaire robuste, le **Random Forest Regressor**, capable de capturer la relation entre les variables d'entrée et la cible pondérale.

In [5]:
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Modèle RandomForestRegressor entraîné avec succès !")

Modèle RandomForestRegressor entraîné avec succès !


## 5. Évaluation des performances du modèle

Nous mesurons l'exactitude de nos prédictions sur le jeu de test à l'aide de deux métriques clés :

* **R² (Coefficient de détermination)** : Il mesure la proportion de la variance de la variable cible qui est expliquée par les variables explicatives du modèle. Un $R^2$ proche de 1 indique un modèle extrêmement précis.
* **MAE (Mean Absolute Error)** : Il représente l'écart absolu moyen entre les prédictions et les valeurs réelles, exprimé dans l'unité de la cible (grammes).

In [6]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Coefficient de détermination R² : {r2:.4f}")
print(f"Erreur Absolue Moyenne MAE : {mae:.2f} g")

Coefficient de détermination R² : 0.9852
Erreur Absolue Moyenne MAE : 2.15 g


## 6. Sauvegarde du modèle entraîné

Pour pouvoir réutiliser ce modèle au sein de notre API de production FastAPI, nous l'exportons de manière robuste au format Joblib sous le nom `nutri_predict_rf_model.joblib`.

In [7]:
import os

OUTPUT_DIR = "../models/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
model_path = os.path.join(OUTPUT_DIR, "nutri_predict_rf_model.joblib")

joblib.dump(model, model_path)
print(f"Modèle sauvegardé avec succès à l'emplacement : {model_path}")

Modèle sauvegardé avec succès à l'emplacement : ../models/nutri_predict_rf_model.joblib
